# 01 — Understanding the sources

**Class 4 deliverable.** Map business questions to the information required, to the
systems that hold it. Establish ownership, grain and gaps *before* writing any
pipeline code.

The business problem: **NYC TLC operations wants to know where and when taxi trips
are slowest, so it can target curb, signal and dispatch interventions.**

The KPI: **congested trip rate** — the share of valid trips averaging under 8 mph.

In [1]:
import sys, json, duckdb, pandas as pd
sys.path.insert(0, '../src')
from pipeline.config import load_config
cfg = load_config('../config/pipeline.yaml')
con = duckdb.connect()
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 50)
RAW, MARTS, STG = '../data/raw', '../data/marts', '../data/staging'

## 1. Question → information → source

| Business question | Information needed | Source | Grain |
|---|---|---|---|
| How much service runs slowly? | trip start/end time, distance | TLC trip extracts | 1 row / trip |
| *Where* is it slow? | pickup/dropoff zone → borough | TLC extracts + zone lookup | trip; zone |
| *When* is it slow? | pickup timestamp | TLC extracts | trip |
| Weather, or structural? | hourly precipitation | Open-Meteo API | 1 row / hour |
| Are airport runs different? | airport zone membership | zone lookup | zone |
| Can we trust the answer? | rejected-record counts | the pipeline itself | rule/fleet/month |

Full reasoning in [`docs/source_map.md`](../docs/source_map.md).

## 2. The two trip sources are *not* the same schema

In [2]:
y = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{RAW}/trips/yellow/2024-01.parquet')").df()
g = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{RAW}/trips/green/2024-01.parquet')").df()
ycols, gcols = set(y.column_name), set(g.column_name)
print(f"yellow: {len(ycols)} columns | green: {len(gcols)} columns")
print("\nonly in yellow:", sorted(ycols - gcols))
print("only in green :", sorted(gcols - ycols))
print("\nshared        :", len(ycols & gcols), "columns")

yellow: 19 columns | green: 20 columns

only in yellow: ['Airport_fee', 'tpep_dropoff_datetime', 'tpep_pickup_datetime']
only in green : ['ehail_fee', 'lpep_dropoff_datetime', 'lpep_pickup_datetime', 'trip_type']

shared        : 16 columns


This is the first real FDE finding: the two fleets describe the *same real-world
event* with different column names. `tpep_*` vs `lpep_*` timestamps; green adds
`ehail_fee`/`trip_type`; yellow has an airport fee column that green lacks — and
whose capitalisation (`Airport_fee` vs `airport_fee`) varies by month.

Treating these as one source without reconciliation would silently produce NULL
timestamps for one fleet. The contract lives in
[`src/pipeline/schema.py`](../src/pipeline/schema.py) and is enforced by a
completeness check, so a future schema change fails loudly instead of quietly.

## 3. Grain and volume — the fleets are wildly asymmetric

In [3]:
rows = []
for fleet in ['yellow', 'green']:
    n = con.sql(f"SELECT count(*) FROM read_parquet('{RAW}/trips/{fleet}/2024-01.parquet')").fetchone()[0]
    rows.append({'fleet': fleet, 'rows_2024_01': n})
df = pd.DataFrame(rows)
df['share_pct'] = (100 * df.rows_2024_01 / df.rows_2024_01.sum()).round(2)
display(df)
print("Green is a small fleet - but it serves the outer boroughs yellow cabs cannot,")
print("so it is retained for coverage, not volume.")

,fleet,rows_2024_01,share_pct
0,yellow,2964624,98.13
1,green,56551,1.87


Green is a small fleet - but it serves the outer boroughs yellow cabs cannot,
so it is retained for coverage, not volume.


## 4. Confirming the grain: one row = one trip?

In [4]:
# If the grain is truly one row per trip, an exact duplicate should be impossible.
dupes = con.sql(f'''
    SELECT count(*) - count(DISTINCT (tpep_pickup_datetime, tpep_dropoff_datetime,
           PULocationID, DOLocationID, trip_distance, total_amount, VendorID))
    FROM read_parquet('{RAW}/trips/yellow/2024-01.parquet')
''').fetchone()[0]
print(f"exact duplicate rows in yellow 2024-01: {dupes}")
print("Grain confirmed: one row per completed trip. No dedup step is warranted.")

exact duplicate rows in yellow 2024-01: 0
Grain confirmed: one row per completed trip. No dedup step is warranted.


## 5. The reference source: what `LocationID` actually means

In [5]:
z = con.sql(f"SELECT * FROM read_csv('{RAW}/reference/taxi_zone_lookup.csv')").df()
print(f"{len(z)} zones")
display(z.groupby('Borough').size().rename('zones').reset_index())
print("\nSentinel values that are codes, not places:")
display(z[z.LocationID.isin([264, 265])])

265 zones


,Borough,zones
0,Bronx,43
1,Brooklyn,61
2,EWR,1
3,Manhattan,69
4,N/A,1
5,Queens,69
6,Staten Island,20
7,Unknown,1



Sentinel values that are codes, not places:


,LocationID,Borough,Zone,service_zone
263,264,Unknown,N/A,N/A
264,265,N/A,Outside of NYC,N/A


Zones **264 (Unknown)** and **265 (Outside of NYC)** are sentinel codes. Treating
them as ordinary zones would invent a phantom "Unknown borough" in every report.
Rule V06 handles them by excluding affected trips from *geographic* metrics only —
the trips themselves are real and stay in citywide figures.

Airports are zones **1 (EWR)**, **132 (JFK)** and **138 (LaGuardia)**.

## 6. The external source: weather (API)

In [6]:
w = json.load(open(f'{RAW}/weather/2024-01.json'))
print("units:", w['hourly_units'])
print("hours returned:", len(w['hourly']['time']))
print("station point:", w['latitude'], w['longitude'], '|', w['timezone'])
print("\nfirst 3 hours:", w['hourly']['time'][:3])

units: {'time': 'iso8601', 'temperature_2m': '°C', 'precipitation': 'mm', 'snowfall': 'cm'}
hours returned: 744
station point: 40.808434 -74.0199 | America/New_York

first 3 hours: ['2024-01-01T00:00', '2024-01-01T01:00', '2024-01-01T02:00']


This is the only source outside TLC's control, and the only one that can answer
*"is the congestion weather-driven or structural?"*. One point (Central Park)
stands in for all five boroughs — an accepted limitation, recorded in
[`docs/known_unknowns.md`](../docs/known_unknowns.md) (A4).

## 7. Gaps — what this data fundamentally cannot answer

| Gap | Consequence for the KPI |
|---|---|
| No route/waypoints | We can rank *where*, never explain *why* without fieldwork |
| Zones, not coordinates | Interventions are zone-level, not street-level |
| **No cancelled or unserved requests** | A zone nobody will serve looks *healthy* — unmet demand is invisible |
| No ride-hail (Uber/Lyft) | Taxis are a biased *sample* of traffic, not a census |
| ~2-month publication lag | A planning instrument, not a live monitor |

The third is the most consequential and is stated prominently wherever the KPI is
published.